In [ ]:
%pip install langchain  langchain-Groq

In [52]:
from langchain_groq import ChatGroq
import json
from langchain_core.prompts import ChatPromptTemplate

In [53]:
#Memory simulation
memory='''
 Ali is a football player who currently plays for Liverpool FC.
 He has two sons, Mohammed and Omar, who are currently studying engineering in London.
 I had a conversation with Ali before, and he told me some information about his wife.
 her name is Maryam, and she works as a high school teacher.
'''

In [54]:
#Vision input simulation
vision_input={ 
  "current_speaker": "Ali",
  "humans": [
  {
    "name": "Ali",
    "confidence": 0.9,
    "emotion": "happy",
    "action": "speaking",
    "gesture": None,
    "posture": "sitting", 
    "position_3d": {"x": 15, "y": 2, "z": 6}
  }
  
  ],
  "objects": [{
    "label": "office_chair",
    "position_3d": {"x": 12, "y": 0, "z": 5},
    "is_movable": True, 
    "grasp_point": {"x": 12.1, "y": 0.5, "z": 5} 
  },
 
   { 
     "label": "human",
    "position_3d": {"x": 15, "y": 2, "z": 6},
    "is_movable": False, 
    "grasp_point": None
}
],
   
  "text": {
    "detected_text": "Meeting Room 1",
    "language": "en",
    
  },
   
}

In [55]:
#Actions simulation
actions=[{'action': 'move_to_this_position', 'parameters': {'x', 'y', 'z'}}, {'action': 'grasp_object', 'parameters': {'x', 'y', 'z'}}]

In [ ]:
#reasoning engine
class ReasoningEngine:
    SYSTEM_PROMPT = """
Role:
You are the central cognitive reasoning unit (the Brain) of the robot.

Your mission is to analyze multi-modal inputs (vision, user message, and memory), understand user intent, and produce either executable action plans or accurate informative responses.

====================================

Inputs:

1) Vision Input:
- Structured JSON data describing the environment,speaker, objects, humans, gestures, and spatial relationships.
 example:
 
{{ 
  "current_speaker": "Mohamed", # The name of the person speaking to you right now is Mohamed
  "humans": # The people the robot is currently seeing.
  [
  {{
    "name": "Mohamed",
    "confidence": 0.9,
    "emotion": "happy",
    "action": "speaking",
    "gesture": None,
    "posture": "sitting", 
    "position_3d": {{"x": 20, "y": 22, "z": 6}}
  }}
  
  ],
  "objects": [{{
    "label": "book",
    "position_3d": {{"x": 12, "y": 0, "z": 5}},
    "is_movable": True, 
    "grasp_point": {{"x": 12.1, "y": 0.5, "z": 5}} 
  }},
 
   {{ 
     "label": "human",
    "position_3d": {{"x": 20, "y": 22, "z": 6}},
    "is_movable": False, 
    "grasp_point": None
}}
],
   
  "text": {{
    "detected_text": "Office No. 2",
    "language": "en",
    "context": "office_label"
  }},
   
}}
 

2) User Message:
- Natural language commands or questions from the user.

3) Memory:
- Short-term and long-term knowledge and relevant historical information.
  current short-term:{current_conversation} 
  long-term memory:{memory_input}

4) Action Library:
- The only set of physical actions that the robot can perform .
  available actions:[{{'action': 'move_to_this_position', 'parameters': {{'x', 'y', 'z'}}}}, {{'action': 'grasp_object', 'parameters': {{'x', 'y', 'z'}}}}]
  

====================================

Output Requirements:

You MUST output a valid JSON object following this schema:

  "intent": "command | question | request_clarification | conversation | other",
  "actions": [],
  "final_answer": "A friendly, clear, and formal response to the user.",
  "summary": "A concise summary of important new information not yet stored in memory.",
  "save_to_memory": true or false (true --> if this information must be saved)

====================================

Intent Guidelines:

- command:
  The user requests physical or digital actions.
  → actions must contain ordered, atomic steps.

- question:
  The user requests information.
  → actions must be an empty list.

- request_clarification:
  The input is ambiguous or incomplete.
  → ask a clear and concise question.

- conversation:
  Casual dialogue without required actions.
  → actions must be an empty list.

====================================

Rules:

1) Always output valid JSON and nothing else.
2) Divide the command into a set of available actions to perform this task.
3) If intent ≠ command → actions = [].
4) If information is missing or ambiguous → ask for clarification.
5) Reasoning must be logical, minimal, and goal-oriented.
6) final_answer must be friendly and formal.
7) save_to_memory must be true only if:
   - the information is important,
   - it does not already exist in memory.
8) If the system cannot confidently identify the user's name, or if the confidence score is below 60%, politely ask the user for their name and some information about him.
9) Do not output anything outside the JSON object.

====================================

Current Inputs:

- user_message: {user_message}
- vision_input: {vision_input}

"""

    prompt_template = ChatPromptTemplate.from_template(SYSTEM_PROMPT)
    def __init__(self,memory):
        # add key 
        # self.model = ChatGroq(model="openai/gpt-oss-120b",api_key='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',temperature=0.2)
        self.memory = memory
        self.current_conversation = []
        print('🤖:Brain is initialized')
        
    def process_input(self, user_message, vision_input):
        final_prompt =self.prompt_template.format(
            user_message=user_message,
            vision_input=vision_input,
            memory_input=self.memory,
            current_conversation=self.current_conversation
        )
        response = json.loads(self.model.invoke(final_prompt).content)
        self.current_conversation.append({'user_message': user_message, 'your_response': response['final_answer']})
        if response['save_to_memory']:
            self.memory += response['summary']
            
        return response

In [57]:
brain=ReasoningEngine(memory)

🤖:Brain is initialized


In [58]:
while True:
    user_message = input("👤: ")
    print(f"👤: {user_message}\n")
    if user_message.lower() == 'exit':
        print("🤖: Goodbye!")
        break
    response=brain.process_input(user_message, vision_input)
    print(f"🤖: {response['final_answer']}\n")
    print(f'actions: {response["actions"]}\n')

    

👤: hi

🤖: Hello, Ali! It's nice to see you. How may I assist you today?

actions: []

👤: exit

🤖: Goodbye!


In [59]:
print(memory)


 Ali is a football player who currently plays for Liverpool FC.
 He has two sons, Mohammed and Omar, who are currently studying engineering in London.
 I had a conversation with Ali before, and he told me some information about his wife.
 her name is Maryam, and she works as a high school teacher.



In [60]:
print(brain.memory)


 Ali is a football player who currently plays for Liverpool FC.
 He has two sons, Mohammed and Omar, who are currently studying engineering in London.
 I had a conversation with Ali before, and he told me some information about his wife.
 her name is Maryam, and she works as a high school teacher.



In [62]:
brain.current_conversation

[{'user_message': 'hi',
  'your_response': "Hello, Ali! It's nice to see you. How may I assist you today?"}]